In [35]:
import math

# Precomputed constants
R = 2.8614**2 + 126.994**2       # ~ 16135.95
C = 133.3**2 + 0.5**2            # ~ 17769.14

def forward_kinematics(theta1, theta2, theta3):
    """
    Forward kinematics for the given joint angles (in radians).
    Returns the homogeneous transformation matrix ^0_T_ee as a list of lists.
    """
    c1, s1 = math.cos(theta1), math.sin(theta1)
    c2, s2 = math.cos(theta2), math.sin(theta2)
    c23, s23 = math.cos(theta2 + theta3), math.sin(theta2 + theta3)

    # Position terms
    px = c1 * (2.8614 * s23 - 126.994 * c23 - 133.3 * c2 + 0.5 * s2 + 1.3) - 0.2645 * s1
    py = s1 * (2.8614 * s23 - 126.994 * c23 - 133.3 * c2 + 0.5 * s2 + 1.3) + 0.2645 * c1
    pz = 126.994 * s23 + 2.8614 * c23 + 0.5 * c2 + 133.3 * s2 + 95
    # Rotation matrix components
    R00, R01, R02 = c1 * c23, -s1, c1 * s23
    R10, R11, R12 = s1 * c23, c1, s1 * s23
    R20, R21, R22 = -s23, 0, c23

    T = [
        [R00, R01, R02, px],
        [R10, R11, R12, py],
        [R20, R21, R22, pz],
        [0,   0,   0,   1]
    ]
    return T


def inverse_kinematics(x, y, z):
    """
    Inverse kinematics for given end-effector position (x, y, z).
    Returns a list of possible (theta1, theta2, theta3) solutions in radians.
    """
    solutions = []

    # Step 1: theta1
    rho_sq = x**2 + y**2
    if rho_sq < 0.2645**2:
        return []  # No real solution for theta1

    base_angle = math.atan2(x, -y)
    delta_angle = math.acos(-0.2645 / math.sqrt(rho_sq))

    theta1_candidates = [
        base_angle + delta_angle,
        base_angle - delta_angle
    ]

    # Step 2: theta2
    for theta1 in theta1_candidates:
        c1, s1 = math.cos(theta1), math.sin(theta1)
        U = c1 * x + s1 * y - 1.3
        V = z - 95
        M = 133.3 * U - 0.5 * V
        N = -0.5 * U - 133.3 * V
        L = 0.5 * (R - C - U**2 - V**2)

        if M**2 + N**2 < L**2:
            continue  # No real solution for theta2 for this theta1

        base_angle2 = math.atan2(N, M)
        delta_angle2 = math.acos(L / math.sqrt(M**2 + N**2))

        theta2_candidates = [
            base_angle2 + delta_angle2,
            base_angle2 - delta_angle2
        ]

        # Step 3: theta3
        for theta2 in theta2_candidates:
            c2, s2 = math.cos(theta2), math.sin(theta2)
            P = U + 133.3 * c2 - 0.5 * s2
            Q = V - 0.5 * c2 - 133.3 * s2
            s23 = (2.8614 * P + 126.994 * Q) / R
            c23 = (-126.994 * P + 2.8614 * Q) / R

            theta23 = math.atan2(s23, c23)
            theta3 = theta23 - theta2

            solutions.append((theta1, theta2, theta3))

    return solutions

In [36]:
import numpy as np
import pandas as pd

def forward_kinematics_np(theta1, theta2, theta3):
    # All inputs are 1D arrays of equal length
    c1, s1 = np.cos(theta1), np.sin(theta1)
    c2, s2 = np.cos(theta2), np.sin(theta2)
    c23, s23 = np.cos(theta2 + theta3), np.sin(theta2 + theta3)

    # Rotation matrix components
    n_x = c1 * c23
    n_y = s1 * c23
    n_z = -s23

    o_x = -s1
    o_y = c1
    o_z = np.zeros_like(theta1)

    a_x = c1 * s23
    a_y = s1 * s23
    a_z = c23

    # Position
    px = c1 * (2.8614 * s23 - 126.994 * c23 - 133.3 * c2 + 0.5 * s2 + 1.3) - 0.2645 * s1
    py = s1 * (2.8614 * s23 - 126.994 * c23 - 133.3 * c2 + 0.5 * s2 + 1.3) + 0.2645 * c1
    pz = 126.994 * s23 + 2.8614 * c23 + 0.5 * c2 + 133.3 * s2 + 95

    # Stack into (N, 12) array
    return np.column_stack([n_x, n_y, n_z,
                            o_x, o_y, o_z,
                            a_x, a_y, a_z,
                            px, py, pz])

def generate_data():
    step = 0.05
    angles = np.arange(-np.pi, np.pi, step)  # radians directly
    t1, t2, t3 = np.meshgrid(angles, angles, angles, indexing='ij')
    
    # Flatten all combinations
    t1f, t2f, t3f = t1.ravel(), t2.ravel(), t3.ravel()

    # Vectorized FK
    fk_vals = forward_kinematics_np(t1f, t2f, t3f)

    # Combine into DataFrame
    df = pd.DataFrame(
        np.column_stack([t1f, t2f, t3f, fk_vals]),
        columns=['theta1', 'theta2', 'theta3',
                 'n_x', 'n_y', 'n_z',
                 'o_x', 'o_y', 'o_z',
                 'a_x', 'a_y', 'a_z',
                 'x', 'y', 'z']
    )
    df.to_csv('kinematics_data.csv', index=False)


In [37]:
generate_data()

In [38]:
sample_angle = math.radians(30), math.radians(45), math.radians(-20)  # Radians
T = forward_kinematics(*sample_angle)
print("Sample angles (radians):", sample_angle)
print("Forward kinematics result (homogeneous transformation matrix):")
for row in T:
    print(np.round(row, 3))

x, y, z = T[0][3], T[1][3], T[2][3]
ik_solutions = inverse_kinematics(x, y, z)
print("\nInverse kinematics solutions (radians):")
for sol in ik_solutions:
    print(np.round(sol, 3))
 

Sample angles (radians): (0.5235987755982988, 0.7853981633974483, -0.3490658503988659)
Forward kinematics result (homogeneous transformation matrix):
[   0.785   -0.5      0.366 -178.958]
[   0.453    0.866    0.211 -103.016]
[ -0.423   0.      0.906 245.874]
[0 0 0 1]

Inverse kinematics solutions (radians):
[0.524 0.463 0.312]
[ 0.524 -5.498  5.934]
[-2.621  2.3    0.398]
[-2.621 -3.576  5.847]


In [40]:
import numpy as np

# Load only the needed columns as a NumPy array
data = np.loadtxt('kinematics_data.csv', delimiter=',', skiprows=1)

# Extract columns by index
# Assuming CSV columns:
# 0:theta1, 1:theta2, 2:theta3, ..., 12:x, 13:y, 14:z
theta_cols = data[:, :3]   # theta1, theta2, theta3
x_vals = data[:, 12]
y_vals = data[:, 13]
z_vals = data[:, 14]

# Find closest point
dx = x_vals - x
dy = y_vals - y
dz = z_vals - z
dist_sq = dx**2 + dy**2 + dz**2
closest_idx = np.argmin(dist_sq)

# Get the row
theta1, theta2, theta3 = theta_cols[closest_idx]
x_closest, y_closest, z_closest = x_vals[closest_idx], y_vals[closest_idx], z_vals[closest_idx]

# Round to 10 decimal places
rounding_digit = 3

theta1 = round(theta1, rounding_digit)
theta2 = round(theta2, rounding_digit)
theta3 = round(theta3, rounding_digit)
x_closest = round(x_closest, rounding_digit)
y_closest = round(y_closest, rounding_digit)
z_closest = round(z_closest, rounding_digit)

dx = round(x_closest - x, rounding_digit)
dy = round(y_closest - y, rounding_digit)
dz = round(z_closest - z, rounding_digit)

# Print results
print("\nClosest point in dataset:")
print(f"theta1: {theta1}, theta2: {theta2}, theta3: {theta3}, "
      f"x: {x_closest}, y: {y_closest}, z: {z_closest}")
print("Distance to closest point:", dx, dy, dz)




Closest point in dataset:
theta1: 0.508, theta2: 0.808, theta3: -0.392, x: -179.519, y: -99.738, z: 245.776
Distance to closest point: -0.561 3.278 -0.098
